# 00 — Dataset anatomy (DevSet1010 / TestSet300 / TestSetNew46)
Pre-model audit for metal / nuclear / small: prevalence, overlap, AA enrichment, local context,
clustering, cross-split stability, cached-feature fingerprints. Every table is cached under
`results/ALL_BINDINGS_DATASET_ANATOMY_V1/` (`force=True` recomputes).

In [ ]:
# ============================================================
# BOOTSTRAP — Colab: mount Drive, put pbind on the path
# ============================================================
import sys, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

# Folder of this repo (copy it to Drive once, or git clone it into /content).
REPO = Path("/content/drive/MyDrive/Protein-binding-Phi/protein-binding")
sys.path.insert(0, str(REPO / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)

# Data root (Drive layout). Override with PBIND_BASE for a local mirror.
# import os; os.environ["PBIND_BASE"] = "/content/drive/MyDrive/Protein-binding-Phi"

from pbind.paths import resolve_paths, check_inputs
display(check_inputs(resolve_paths()))

In [ ]:
from pbind.pipelines import open_workspace, anatomy
from pbind.analysis import enrichment_pivot

# with_backend=True also loads GEAR stores for the feature-block fingerprint
ws = open_workspace(task=None, with_backend=True, attach_evo=False, attach_esmif=False)
R = anatomy.dataset_anatomy(ws)

## Core task anatomy / overlap

In [ ]:
display(R["core"].pivot(index="task", columns="dataset", values="residue_prevalence"))
display(R["overlap"])

## AA enrichment per task

In [ ]:
for task in ["metal_binding", "nuclear_binding", "small_binding"]:
    print("###", task); display(enrichment_pivot(R["aa"], task))
display(R["task_similarity"])

## Context, clustering, stability, decision tables

In [ ]:
display(R["seqctx"].assign(abs_smd=lambda d: d.standardized_mean_difference.abs())
        .sort_values("abs_smd", ascending=False).head(30))
display(R["cluster_summary"])
display(R["stability_verdicts"]); display(R["task_distinctiveness"]); display(R["mechanism_hints"])
if "fingerprint" in R:
    display(R["fingerprint"].sort_values(["dataset", "task", "effect_rms"], ascending=[True, True, False]))